# Iris Flower Classification
### CodeAlpha Data Science Internship — Task 1

## 1. Project Introduction

This notebook builds a complete machine learning workflow to classify Iris flowers into one of three species — **Setosa**, **Versicolor**, or **Virginica** — using four simple measurements: sepal length, sepal width, petal length, and petal width.

We use **Logistic Regression**, a simple and beginner-friendly classification algorithm, and evaluate it using standard metrics like accuracy, precision, recall, and F1-score.

## 2. Import Libraries

We import all the libraries needed for data handling, visualization, and machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, precision_score,
                             recall_score, f1_score)

sns.set_style('whitegrid')
%matplotlib inline

## 3. Load Dataset

We load the Iris dataset from `dataset/Iris.csv`. The file has an `Id` column (just a row number) and long column names, so we drop `Id`, rename the columns to simple names, and shorten the species names (for example `Iris-setosa` becomes `Setosa`).

In [ ]:
feature_columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

# Read the dataset file
raw_df = pd.read_csv('dataset/Iris.csv')

# Standardize: drop the 'Id' column, rename columns, shorten species names
df = raw_df.rename(columns={
    'SepalLengthCm': 'sepal_length',
    'SepalWidthCm': 'sepal_width',
    'PetalLengthCm': 'petal_length',
    'PetalWidthCm': 'petal_width',
    'Species': 'species',
})
df = df[feature_columns + ['species']]
df['species'] = df['species'].replace({
    'Iris-setosa': 'Setosa',
    'Iris-versicolor': 'Versicolor',
    'Iris-virginica': 'Virginica',
})

df.head()

## 4. Understand Dataset

Before doing anything else, let's understand the shape and structure of our data.

In [ ]:
print('Number of rows:', df.shape[0])
print('Number of columns:', df.shape[1])
print('\nColumn names:', list(df.columns))
print('\nData types:')
print(df.dtypes)

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
df.describe()

In [ ]:
df['species'].value_counts()

## 5. Data Cleaning

Even though the built-in Iris dataset is very clean, we still apply the same validation and cleaning steps we would use on any real-world dataset:

- Remove duplicate rows
- Remove rows with missing values
- Remove rows with zero or negative measurements (physically impossible)
- Remove rows with unrealistic (out-of-range) measurements
- Keep only valid species labels

This makes the project realistic and demonstrates good data science practice, even when the raw data happens to already be clean.

In [ ]:
feature_ranges = {
    'sepal_length': (3.0, 9.0),
    'sepal_width': (1.5, 5.5),
    'petal_length': (0.5, 8.0),
    'petal_width': (0.05, 3.5),
}

df_clean = df.copy()
before = df_clean.shape[0]

df_clean = df_clean.dropna(subset=feature_columns + ['species'])
df_clean = df_clean.drop_duplicates()

for col in feature_columns:
    df_clean = df_clean[df_clean[col] > 0]

for col in feature_columns:
    low, high = feature_ranges[col]
    df_clean = df_clean[(df_clean[col] >= low) & (df_clean[col] <= high)]

valid_species = {'Setosa', 'Versicolor', 'Virginica'}
df_clean = df_clean[df_clean['species'].isin(valid_species)]
df_clean = df_clean.reset_index(drop=True)

after = df_clean.shape[0]
print(f'Rows before cleaning: {before}')
print(f'Rows after cleaning: {after}')

## 6. Exploratory Data Analysis

Let's explore the relationships between features and species.

In [ ]:
df_clean.groupby('species')[feature_columns].mean()

## 7. Data Visualization

We create a few clear, meaningful charts: class distribution, feature distributions, and feature relationships.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_clean, x='species', hue='species',
              order=sorted(df_clean['species'].unique()), palette='viridis', legend=False)
plt.title('Species Class Distribution')
plt.xlabel('Species')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('images/class_distribution.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, col in zip(axes.flatten(), feature_columns):
    sns.histplot(data=df_clean, x=col, hue='species', kde=True, ax=ax, palette='Set2')
    ax.set_title(f'Distribution of {col.replace("_", " ").title()}')
plt.tight_layout()
plt.savefig('images/feature_distributions.png', dpi=150)
plt.show()

In [ ]:
sns.pairplot(df_clean, hue='species', palette='husl')
plt.savefig('images/feature_relationships.png', dpi=150)
plt.show()

## 8. Prepare Features and Target

We separate the input features (X) from the target label (y).

In [ ]:
X = df_clean[feature_columns]
y = df_clean['species']

print('Features shape:', X.shape)
print('Target shape:', y.shape)

## 9. Train/Test Split

We split the data into 80% for training and 20% for testing, using a fixed `random_state` so the results are reproducible every time this notebook runs.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 10. Feature Scaling

Logistic Regression works better when features are on a similar scale. We use `StandardScaler`, combined with the classifier inside a single `Pipeline`, so scaling is applied consistently during both training and prediction.

In [ ]:
# Feature scaling is handled inside the Pipeline in the next step,
# so training and prediction always use the same scaling.
print('StandardScaler will be applied inside the Pipeline (next step).')

## 11. Train Logistic Regression

We build a Pipeline combining `StandardScaler` and `LogisticRegression`, then train it on the training data.

In [ ]:
model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=200, random_state=42)),
])

model.fit(X_train, y_train)
print('Model trained successfully.')

## 12. Make Predictions

We use the trained model to predict species for the test set.

In [ ]:
y_pred = model.predict(X_test)
pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred}).head(10)

## 13. Evaluate Model

We calculate accuracy, precision, recall, and F1-score. All values here are computed live from the model's actual predictions — nothing is hardcoded.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')

## 14. Confusion Matrix

The confusion matrix shows exactly which species were predicted correctly, and which were confused with each other.

In [ ]:
labels_sorted = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_sorted, yticklabels=labels_sorted)
plt.xlabel('Predicted Species')
plt.ylabel('Actual Species')
plt.title('Confusion Matrix - Logistic Regression')
plt.tight_layout()
plt.savefig('images/confusion_matrix.png', dpi=150)
plt.show()

## 15. Classification Report

The classification report shows precision, recall, and F1-score for each individual species.

In [ ]:
print(classification_report(y_test, y_pred))

## 16. Prediction Example

Let's try predicting the species for a single, new flower measurement.

In [ ]:
sample = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=feature_columns)
predicted_species = model.predict(sample)[0]
probabilities = model.predict_proba(sample)[0]

print('Sample measurements:', sample.values.tolist())
print('Predicted species:', predicted_species)
print('\nPrediction probabilities:')
for label, prob in zip(model.classes_, probabilities):
    print(f'  {label}: {prob * 100:.2f}%')

In [ ]:
# Save the trained model so it can be used by the Streamlit app (app.py)
joblib.dump(model, 'model/iris_model.pkl')
print('Model saved to model/iris_model.pkl')

## 17. Conclusion

In this notebook, we:

1. Loaded the Iris dataset from the CSV file (`dataset/Iris.csv`).
2. Validated and cleaned the data.
3. Explored and visualized the dataset.
4. Trained a Logistic Regression model inside a scaling Pipeline.
5. Evaluated the model using accuracy, precision, recall, F1-score, and a confusion matrix.
6. Saved the trained model for use in the Streamlit web application (`app.py`).

This project demonstrates a complete, end-to-end machine learning workflow that is simple enough for a beginner to understand and explain, while still following good data science practices.